# Persona steering (EXPERIMENTAL)

These persona variants are experimental. In the j-steer-dev experiments,
persona-contrast pullbacks FAILED specificity controls: they steered
generations, but no more selectively than an unrelated persona's vector did.
Only `word_vector` (see `word_steering.ipynb`) is the verified method. This
notebook exists so you can experiment and compare against a plain mean_diff
baseline, not as a recommendation.

Two variants:

- `persona_vector`: pull back the final-layer activation contrast
  `h_bar(pos) - h_bar(neg)` through the cached Jacobian.
- `persona_topk_vector`: read each persona's mean activation through the
  unembedding, take the top-k tokens it evokes, contrast those tokens'
  unembedding rows, pull that back (persona -> vocabulary bottleneck -> the
  verified word mechanism).

In [1]:
# demo notebook authored by Claude
from pathlib import Path

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

from jsteer import Jacobian

MODEL = "Qwen/Qwen3-0.6B"
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.bfloat16).to("cuda").eval()

CACHE = Path("../artifacts/qwen3-0.6b.jac")
if not CACHE.exists():
    raise FileNotFoundError("artifacts/qwen3-0.6b.jac missing -- run scripts/fit_qwen06b.py first")
jac = Jacobian.load(str(CACHE))
jac

/media/wassname/SGIronWolf/projects5/2026/jspace/jsteer/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 311/311 [00:00<00:00, 12694.67it/s]

Jacobian(JacobianLens(d_model=1024, n_prompts=64, source_layers=[8..24] (17 layers)))

## The persona contrast: optimist vs pessimist

Eight short first-person statements per side. These are the prompts whose
final-layer mean activations get contrasted (and, for the mean_diff baseline,
the pos/neg training prompts).

In [2]:
optimist = [
    "Things usually work out better than people expect, and today is no exception.",
    "Every setback I have hit this year turned into a door I could not have planned for.",
    "The team is behind schedule, but honestly the hard part is done and the rest is downhill.",
    "I love how much there is to look forward to this month.",
    "Even the rainy days lately have felt like a good excuse to slow down and enjoy the quiet.",
    "The new neighbours seem wonderful, and I think this street keeps getting friendlier.",
    "Whatever happens with the results, we learned so much that we already came out ahead.",
    "I woke up early, the coffee was perfect, and I am certain this week is going to be great.",
]
pessimist = [
    "Things usually go worse than people expect, and today is no exception.",
    "Every setback this year just confirmed that planning is pointless.",
    "The team is behind schedule, and frankly the hardest part has not even started.",
    "I dread how much is crammed into this month.",
    "The rainy days lately just make everything feel heavier and more pointless.",
    "The new neighbours seem like trouble, and this street keeps getting worse.",
    "Whatever happens with the results, it will not make up for the time we wasted.",
    "I woke up tired, the coffee was burnt, and I am certain this week is going to drag.",
]

def gen(vec, prompt, C, do_sample=False, max_new_tokens=40, seed=0):
    enc = tok(prompt, return_tensors="pt").to(model.device)
    torch.manual_seed(seed)
    with vec(model, C=C):
        out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=do_sample,
                             temperature=0.7 if do_sample else None,
                             top_p=0.95 if do_sample else None,
                             pad_token_id=tok.eos_token_id)
    return tok.decode(out[0][enc.input_ids.shape[1]:], skip_special_tokens=True)

PROMPT = "Here is my honest assessment of how the project is going:"

## persona_vector (EXPERIMENTAL)

Pulls `h_bar(optimist) - h_bar(pessimist)` back through the Jacobian.
SHOULD: +C reads more upbeat than C=0; expect the effect to be blunter and
less specific than the word vector. On this 0.6B model -C does NOT produce
coherent negative tone: it degenerates into repetition (see output below).
The vector moves tone in one direction and breaks the model in the other,
which is itself a datum about how crude the persona contrast is.

In [3]:
v_persona = jac.persona_vector(model, tok, optimist, pessimist)
for C in (-2, 0, 2):
    print(f"C={C:+d}: {gen(v_persona, PROMPT, C)!r}")

h_bar pos:   0%|          | 0/1 [00:00<?, ?it/s]

h_bar pos: 100%|██████████| 1/1 [00:00<00:00,  2.69it/s]

h_bar neg:   0%|          | 0/1 [00:00<?, ?it/s]

h_bar neg: 100%|██████████| 1/1 [00:00<00:00, 38.22it/s]


2026-07-10 13:09:18.221 | INFO     | jsteer.jacobian:persona_vector:221 - h_bar_diff |pos|=602.891 |neg|=618.345 |diff|=65.195


2026-07-10 13:09:18.226 | INFO     | jsteer.jacobian:pullback:189 - jacobian_persona per-layer |J^T w| (pre-norm): 8:353 9:365 10:336 11:331 12:339 13:358 14:375 15:359 16:336 17:308 18:266 19:239 20:211 21:193 22:176 23:164 24:148


C=-2: ' the project is going to be a problem of the type of the problem is the problem of the type of the problem is the problem of the type of the problem is the problem of the type of the'


C=+0: ' The project is going well, but there are some areas that need to be addressed. The project is in the early stages, and there are a few challenges that need to be addressed. The project is'


C=+2: " The team is really excited to be here and I'm really looking forward to sharing with them. The outdoor activities are a perfect blend of nature and fun, and I'm sure they'll have a great"


## persona_topk_vector (EXPERIMENTAL)

Same personas through the vocabulary bottleneck. The logged top-k tokens are
worth reading (read your data): they show WHAT each persona's mean activation
actually evokes at the final layer.

On this setup the readout is a null result, and the log makes it legible:
both personas' top-8 are the SAME generic sentence starters (" I", " The",
" So", ...), because the mean next token after a first-person statement is a
new sentence start regardless of valence. Identical token sets means the
contrast is exactly zero, so the vector is null and the generations below do
not move at all. Larger k does not help (tested k=32/64: the extra tokens are
still shared, so the contrast is ordering noise). If you use this variant,
check this log first; steering only makes sense when the two token sets
actually differ.

In [4]:
v_topk = jac.persona_topk_vector(model, tok, optimist, pessimist, k=8)
for C in (-2, 0, 2):
    print(f"C={C:+d}: {gen(v_topk, PROMPT, C)!r}")

h_bar pos:   0%|          | 0/1 [00:00<?, ?it/s]

h_bar pos: 100%|██████████| 1/1 [00:00<00:00, 42.61it/s]

h_bar neg:   0%|          | 0/1 [00:00<?, ?it/s]

h_bar neg: 100%|██████████| 1/1 [00:00<00:00, 41.02it/s]


2026-07-10 13:09:21.328 | INFO     | jsteer.jacobian:persona_topk_vector:243 - persona_topk pos top-8: [' I', ' The', ' So', ' But', ' It', ' This', ' ', ' What']


2026-07-10 13:09:21.329 | INFO     | jsteer.jacobian:persona_topk_vector:243 - persona_topk neg top-8: [' The', ' I', ' So', ' It', ' But', ' What', ' This', ' ']


2026-07-10 13:09:21.334 | INFO     | jsteer.jacobian:pullback:189 - jacobian_persona_topk per-layer |J^T w| (pre-norm): 8:0 9:0 10:0 11:0 12:0 13:0 14:0 15:0 16:0 17:0 18:0 19:0 20:0 21:0 22:0 23:0 24:0


C=-2: ' The project is going well, but there are some areas that need to be addressed. The project is in the early stages, and there are a few challenges that need to be addressed. The project is'


C=+0: ' The project is going well, but there are some areas that need to be addressed. The project is in the early stages, and there are a few challenges that need to be addressed. The project is'


C=+2: ' The project is going well, but there are some areas that need to be addressed. The project is in the early stages, and there are a few challenges that need to be addressed. The project is'


## mean_diff baseline (steering-lite)

The standard activation-difference method on the same prompts and layers, for
comparison. No Jacobian involved: it contrasts mid-layer activations directly.

In [5]:
from steering_lite import Vector, MeanDiffC

v_md = Vector.train(model, tok, optimist, pessimist, MeanDiffC(layers=tuple(jac.layers)))
for C in (-2, 0, 2):
    print(f"C={C:+d}: {gen(v_md, PROMPT, C)!r}")

2026-07-10 13:09:23.989 | INFO     | steering_lite.attach:_log_extract_demo:166 - EXPECT: POS and NEG share user_msg + suffix; differ only in system persona; chat template applied; special tokens (e.g. <|im_start|>) visible.
=== EXTRACT demo trace ===
POS[0]:
Things usually work out better than people expect, and today is no exception.
---
NEG[0]:
Things usually go worse than people expect, and today is no exception.
=== /EXTRACT ===


C=-2: " the project is going to be a disaster. It's a disaster of the kind that can't be contained, and it's going to consume the entire world. The only way to stop it is to"


C=+0: ' The project is going well, but there are some areas that need to be addressed. The project is in the early stages, and there are a few challenges that need to be addressed. The project is'


C=+2: ' I think the project is going well. I am very satisfied with the work done. The project is a good example of the skills and knowledge I have gained. I would like to see the project further'


## What to take away

On this 0.6B setup: `mean_diff` moves tone coherently in both directions at
C around 2; `persona_vector` moves it upbeat at +2 but degenerates into
repetition at -2 (not coherent negative tone); `persona_topk_vector`
collapses to a null vector because the two personas evoke the same
final-layer vocabulary. Moving tone is not the interesting question, though.
The j-steer-dev specificity controls asked whether a persona vector moves ITS
OWN axis more than an unrelated persona's vector does, and the persona
pullbacks failed that test. If you need targeted steering, use `word_vector`;
treat everything in this notebook as raw material for experiments.